In [0]:
from pyspark.sql import *
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
%sql
use catalog lendingclub;
create schema if not exists silver_enriched;
use schema silver_enriched;
select current_catalog(), current_schema();

In [0]:
display(spark.sql('describe extended silver_cleaned.customers'))

In [0]:
#Ideally we should have 1 member_id in table. validate if we have more 
#customer_data

spark.sql('''
        select member_id, count(*) as total_count
        from silver_cleaned.customers
        group by member_id
        order by total_count desc''').show()

In [0]:
bad_cust_df = spark.sql('''
                        select member_id
                        from (select member_id, count(*) as total_count
                        from silver_cleaned.customers
                        group by member_id
                        having total_count > 1)''')

In [0]:
bad_cust_df.count()

In [0]:
dbutils.fs.rm("/Volumes/lendingclub/storagelocation/bad_data/Customer/", recurse=True)

In [0]:
bad_cust_df.write.format('delta').mode('overwrite').save('/Volumes/lendingclub/storagelocation/bad_data/Customer/')

##Seggregate bad data 

In [0]:
bad_cust_df.createOrReplaceTempView('bad_data')

In [0]:
#seggregate bad data from actual cleaned dataset

customer_df = spark.sql('''
                        select * from silver_cleaned.customers 
                        where member_id not in 
                        (select member_id from bad_data)''')

In [0]:
dbutils.fs.rm('/Volumes/lendingclub/storagelocation/final_cleaned/Customer/', recurse=True)

In [0]:
customer_df.write.format('delta').mode('overwrite').save('/Volumes/lendingclub/storagelocation/final_cleaned/Customer/')

In [0]:
%sql
create or replace table silver_enriched.customers
as
select * 
from delta.`/Volumes/lendingclub/storagelocation/final_cleaned/Customer/`

In [0]:
#check  if there is any member_id is repeating

spark.sql('''select member_id, count(*) as total
          from silver_enriched.customers
          group by member_id 
          order by total desc''').show()